In [3]:
import onnx
import copy
import numpy as np
import onnxruntime as ort

In [12]:
model = onnx.load("networks/VertCAS_pra02_v4_45HU_200.onnx")

In [13]:
weight_matrix = copy.deepcopy(model.graph.initializer[0])

In [14]:
weight_matrix

dims: 45
dims: 4
data_type: 1
name: "W0"
raw_data: "\004s\024@Q\335|\276&\032\204>n\031\250\276\274\350\317\277\r\177\327\274\263\302#=hwx\277+M\352?\350\025\377>!w\371\276\227\324\273\273\351\361\207?\365\327\013=\006n\223\274iC4\274@\373\007@6\362\357\274\027\350$\274\337\372$\277I\301\236=\017\270\246>s\031\322<\333\0316\276u\260\026@\304\262\t\276U\242\014>W[\311\2767S\321>\311:<>-?@\276\244\247x\277L\374\211\276$\271\214>{\026\304\276\207\307\251=\\\003\323?8@\272<4b\213<$G\332=\364ut\2748-\214\277C7{<\265:5;\346\314B\277\020Y\234>Z\241h\2767\337x>b\370\022\300\200\265\202>\241\353L\275\036\252d=\253\377\250\275\343\372;?L4\220>p\016\025;\003*w<H\371\231\277\177xL\274\331<3<X\255\320\277K\345\365>\3708\373\276\244\214\030>_\007\206?\354j\302\276\002\233\363>\246H\226>_\'e?\2661\326=\211Z_\275Llb\277v59>\307\325H\276\211@\215\276.\215\302\273t\244\034\301\252\273\303\267Q<y\270U\354\232;i\251p@\341\271j:UTu\273.\334\236\274\215\240\214\275\375\325\306;\236}e\274~\017\007<\262\327\r

In [15]:
arr=np.zeros((4,4+9))
arr[0,0]=1
arr[1,1]=1
arr[2,2]=1
arr[3,3]=1
initializer_tensor = onnx.helper.make_tensor(
name="WInit",
data_type=onnx.TensorProto.FLOAT,
dims=arr.shape,
vals=arr.flatten().tolist())

In [16]:
init_layer = onnx.helper.make_node(
    "MatMul",
    inputs=["WInit", "X"],
    outputs=["XInit"]
)

In [17]:
model.graph.node.insert(0,init_layer)
model.graph.node[1].input[1] = "XInit"
model.graph.initializer.insert(0,initializer_tensor)

In [18]:
model.graph.input[0].type.tensor_type.shape.dim[0].dim_value=13

In [19]:
onnx.save(model, "networks/VertCAS_pra02_v4_45HU_200_FLOAT.onnx")

In [48]:
ort_sess_orig = ort.InferenceSession("networks/VertCAS_pra02_v4_45HU_200.onnx")
ort_sess_float = ort.InferenceSession("networks/VertCAS_pra02_v4_45HU_200_FLOAT.onnx")
x = np.random.uniform(size=4)
x = np.array(x,dtype=np.float32)
eps = np.zeros(9)
print(x)
print(eps)
x_float = np.concatenate([x,eps],dtype=np.float32)
print(x_float)
print(ort_sess_orig.run(None, {'X': x}))
print(ort_sess_float.run(None, {'X': x_float}))

[0.8541019 0.5868536 0.2756361 0.398333 ]
[0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0.8541019 0.5868536 0.2756361 0.398333  0.        0.        0.
 0.        0.        0.        0.        0.        0.       ]
[array([ 0.18209441,  0.03152712, -0.2330774 ,  0.00877565, -0.2548827 ,
       -0.3108577 , -0.5763194 , -0.2980822 , -0.5366758 ], dtype=float32)]
[array([ 0.18209441,  0.03152712, -0.2330774 ,  0.00877565, -0.2548827 ,
       -0.3108577 , -0.5763194 , -0.2980822 , -0.5366758 ], dtype=float32)]


In [21]:
model = onnx.load("networks/VertCAS_pra03_v4_45HU_200.onnx")

In [22]:
weight_matrix = copy.deepcopy(model.graph.initializer[0])

In [23]:
weight_matrix

dims: 45
dims: 4
data_type: 1
name: "W0"
raw_data: "@jq\300\313\267\343\2745\336,\2759`\247\276\273\231\201>\002\274\265\277Yt\206\274\024\272&\274\035wj@\034x\245=\341%\330\275\370\341\360\276)>N?\204bk\276G\314\214>\'\244\365>\265\372\252<\313/;\277\254}\r;X\234\240<\246\355e\300\340\365\221>t\265\235\276g(~\276\252`\200\277YD~\275^\234h\276\334\345\274=\310^;\300\343\251\027>\320C\r\276\205d\371;\325@\303>w\363\274\2774\016$:h\253\024\274\302\206\333\277\034#\t>=\362\257\276n5\266=\304\231\013\300\321\271\221\272\355)\t\276\267\024\306<\236{Y\300\016\3640\275\345X\350\274\257\356\010>ZG\211\277s\203\031\277H\207\347\275\220\276\211<\273C\272\276yY\037\277\205\351\333\275\313\347\246=;\367\203\274(}\341\275\006I\277>zj\332<\241\203^>\300\243\234\275u?\262=\272Nk\277eS\014\300\n\212Z\275\354\277N=\350\207\361=\256G\017\300[D\244=\334\337\217\275\243@\207\277\337\235\301\275\270\036\241?\024\315\346;\273]\020<\217\302Y\300\316q\356=\234O\335\275O\0139=\207\334\334\277\017\235\376=o*\32

In [24]:
arr=np.zeros((4,4+9))
arr[0,0]=1
arr[1,1]=1
arr[2,2]=1
arr[3,3]=1
initializer_tensor = onnx.helper.make_tensor(
name="WInit",
data_type=onnx.TensorProto.FLOAT,
dims=arr.shape,
vals=arr.flatten().tolist())

In [25]:
init_layer = onnx.helper.make_node(
    "MatMul",
    inputs=["WInit", "X"],
    outputs=["XInit"]
)

In [26]:
model.graph.node.insert(0,init_layer)
model.graph.node[1].input[1] = "XInit"
model.graph.initializer.insert(0,initializer_tensor)

In [27]:
model.graph.input[0].type.tensor_type.shape.dim[0].dim_value=13

In [28]:
onnx.save(model, "networks/VertCAS_pra03_v4_45HU_200_FLOAT.onnx")

In [4]:
model = onnx.load("networks/acc-2000000-64-64-64-64-retrain-100000-200000-0.9.onnx")

In [5]:
weight_matrix = copy.deepcopy(model.graph.initializer[0])

In [6]:
model.graph.initializer[1]

dims: 64
data_type: 1
name: "extractor.policy_net.0.bias"
raw_data: "\001_l\273j\220+\271\033\204\005\275!\335\212;wh|\275\177\211\257\273D\344`\275\254\256+\275\237\017\030=\332\230\254:rx\";\202HA\275Nj%:l\205L\274\021\322\334\274\361(\021\275\032\334w\273\177\226\354\271\036\365-\273\232\310f<\361\370\235\274\203\257N\273\265(C\274<\206\027\275\235?/\275\000v/\273y$\354\274\367\256\245\272\267\230\310<\r\0211<\247\370\236<i\004\224\275\335\025F\273\261\220\211\275BI\336\273\361sR\275\270V_\275\316VV\272\371\213\024=#\341\307\272\n\374u<6|\025<.\201\213\274\201h\n\273T5\345;\\\020\177<\214\362\334\274v\'\333<\205\255,\275k&\207=\200Hu\275\241\207\317\272?\205\311<\253\230\346\273\340&\225\274\032\2371\274\273\276\307;{\024\263\273G&\t\274m\243(\275\3730\220=\345u\335<-\263\274\274V\245\007\274"

In [7]:
arr=np.zeros((2,2+1))
arr[0,0]=1
arr[1,1]=1
arr = np.array([arr])
initializer_tensor = onnx.helper.make_tensor(
name="WInit",
data_type=onnx.TensorProto.FLOAT,
dims=arr.shape,
vals=arr.flatten().tolist())

In [8]:
init_layer = onnx.helper.make_node(
    "MatMul",
    inputs=["WInit", "input.1"],
    outputs=["XInit"]
)

In [9]:
model.graph.node.insert(0,init_layer)
model.graph.node[1].input[0] = "XInit"
model.graph.initializer.insert(0,initializer_tensor)

In [10]:
model.graph.input[0].type.tensor_type.shape.dim[0].dim_value=3
model.graph.input[0].type.tensor_type.shape.dim.pop(1)

dim_value: 2

In [11]:
onnx.save(model, "networks/acc-2000000-64-64-64-64-retrain-100000-200000-0.9_FLOAT.onnx")

In [16]:
ort_sess_orig = ort.InferenceSession("networks/acc-2000000-64-64-64-64-retrain-100000-200000-0.9.onnx")
ort_sess_float = ort.InferenceSession("networks/acc-2000000-64-64-64-64-retrain-100000-200000-0.9_FLOAT.onnx")
x = np.random.uniform(size=2)
x = np.array(x,dtype=np.float32)
eps = np.zeros(1)
print(x)
print(eps)
x_float = np.concatenate([x,eps],dtype=np.float32)
print(x_float)
print(ort_sess_orig.run(None, {'input.1': [x]}))
print(ort_sess_float.run(None, {'input.1': x_float}))

[0.352682   0.48537245]
[0.]
[0.352682   0.48537245 0.        ]
[array([[6.0907006]], dtype=float32)]
[array([[6.0907006]], dtype=float32)]
